In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style = "darkgrid")
import numpy as np

In [ ]:
evaluation_df = pd.read_parquet("../data/fraud_evaluation.parquet")

## DID FEATURE ENGINEERING IMPROVE THE MODELS?
For logistic regression, the addition of `policy_duration` and deletion of `incident_date_month` and `incident_date_day` lead to direct model improvement from dataset 1 to dataset 2. However, from dataset 2 to dataset 3, the model became worse, suggesting that `net_capital` does not provide the same predictive power as `capital-loss` and `capital-gain`. In the random forest models, each dataset leads to worse results, suggesting the standard features provide more detail into fraud prediction than the engineered features. In the gradient boosting models, the results remained exactly the same throughout each dataset version, suggesting other features are far more important in predicting fraud in this dataset. 

In [ ]:
best_model_per_version = evaluation_df.loc[[1, 4, 7, 9, 12, 15, 17, 20, 23]]
metrics = ["precision_1", "recall_1", "f1_score_1"]

fig, axes = plt.subplots(1, 3, figsize = (10, 4), constrained_layout = True)
axes = axes.flatten()
plt.suptitle("Fraud Model Metrics per Dataset", fontsize = 14, fontweight = "semibold")

for ax, column in zip(axes, metrics):
    sns.barplot(data = best_model_per_version, x = column, y = "model", hue = "dataset_version", ax = ax)
    ax.set_xlim(min(best_model_per_version[column]) - 0.005, max(best_model_per_version[column]) + 0.005)

axes[2].legend(title = "dataset_version", bbox_to_anchor = (1.05, 1))

for ax in axes[:-1]:
    ax.get_legend().remove()

for ax in axes[1:]:
    ax.yaxis.set_visible(False)

fig.savefig("../figures/fraud_model_metrics_per_dataset.png", dpi = 300)
plt.show()

## BEST MODEL COMPARISON
From the confusion matrices, gradient boosting predicts the most fraud at 45 from 49, while random forest predicts the least at only 40. All models produce similar non-fraud predictions, with logistic regression missing 24 non-fraud incidents and random forest missing 22.

From the bar chart, we can see that the precision for the fraud class is similar for each model. Recall for logistic regression and gradient boosting is similar, with gradient boosting pulling slightly ahead. Random forest achieves considerably lower than the other models. Combining these results, gradient boosting achieves the highest F1-score, with random forest achieving the lowest. For fraud modelling in particular, identifying fraud is more important than misclassifying non-fraud. This means that as long as the precision is not too low, recall is the more prioritised metric. Investigating a potentially fraudulent incident is much easier than missing a truly fraudulent case.

From the ROC curves, both logistic regression and gradient boosting achieve 0.89, outperforming random forest at 0.81. This suggests that the two models are better at distinguishing between fraud and non-fraud incidents across every decision threshold, meaning they rank fraud cases higher than non-fraud cases more consistently than random forest. 

In [ ]:
best_model_comparison = evaluation_df.loc[[9, 4, 23]].reset_index(drop = True)

In [ ]:
xticklabels = ["Predicted Non-fraud", "Predicted Fraud"]
yticklabels = ["Actual Non-fraud", "Actual Fraud"]

fig, axes = plt.subplots(1, 3, figsize = (12, 4), constrained_layout = True)
axes = axes.flatten()
plt.suptitle("Confusion Matrices by Model", fontsize = 14, fontweight = "semibold")

for index, row in best_model_comparison.iterrows():
    matrix = np.array([
        [row["true_positive"], row["false_negative"]],
        [row["false_positive"], row["true_negative"]]
    ])

    sns.heatmap(matrix, cmap = "Blues", annot = True, fmt = '.0f', cbar = False, 
                xticklabels = xticklabels, yticklabels = yticklabels, ax = axes[index])
    axes[index].set_title(f"{row["model"]}")

for ax in axes[1:]:
    ax.yaxis.set_visible(False)

fig.savefig("../figures/fraud_confusion_matrices_by_model.png", dpi = 300)
plt.show()

In [ ]:
metrics = ["precision_1", "recall_1", "f1_score_1"]

fig, axes = plt.subplots(1, 3, figsize = (10, 4), constrained_layout = True)
axes = axes.flatten()
plt.suptitle("Metrics by Model", fontsize = 14, fontweight = "semibold")

for ax, metric in zip(axes, metrics):
    sns.barplot(data = best_model_comparison, x = metric, y = "model", hue = "model", ax = ax)
    ax.set_xlim(min(best_model_comparison[metric]) - 0.01, max(best_model_comparison[metric]) + 0.01)

for ax in axes[1:]:
    ax.yaxis.set_visible(False)

fig.savefig("../figures/fraud_metrics_by_model.png", dpi = 300)
plt.show()

In [ ]:
from sklearn.metrics import auc, RocCurveDisplay

fig, axes = plt.subplots(1, 1, figsize = (12, 5), constrained_layout = True)
plt.suptitle("ROC Curves by Model", fontsize = 14, fontweight = "semibold")

for _, row in best_model_comparison.iterrows():
    roc_auc = auc(row["fpr"], row["tpr"])
    display = RocCurveDisplay(fpr = row["fpr"], tpr = row["tpr"], roc_auc = roc_auc, name = row["model"])
    display.plot(ax = axes)

plt.plot([0, 1], [0, 1], alpha = 0.2, color = 'gray', linestyle = '--')

fig.savefig("../figures/fraud_roc_curves_by_model.png", dpi = 300)
plt.show()

## THE BEST MODEL FEATURES
`incident_severity_Major Damage` was the most influential predictor (0.56), which is appropriate given the distribution of fraud and non-fraud within that feature. `insured_hobbies_chess` and `insured_hobbies_cross-fit` followed closely by 0.23 and 0.14, respectively, which is appropriate given the fraud-to-non-fraud distribution. Every other feature contributes very little to the model's predictions, with its importance score being below 0.02. This means that the gradient boosting model relies heavily on three features. In the top 10 features by feature importance, `net_capital` is the last feature. Although it contributes little to the model's overall performance, this engineered feature captures some information that is used to differentiate fraud from non-fraud incidents. 

In [ ]:
best_model_features = pd.read_parquet("../data/best_classification_model_features.parquet")
best_model_features = best_model_features.rename(columns = {"Column": "Feature"})

fig, axes = plt.subplots(1, 1, figsize = (12, 4), constrained_layout = True)
plt.suptitle("Gradient Boosting - Feature Importance", fontsize = 14, fontweight = "semibold")

best_model_features.sort_values("Feature Importance", ascending = False).head(10).plot(kind = "barh", y = "Feature Importance", 
                                                                                       x = "Feature", ax = axes, legend = False)

fig.savefig("../figures/fraud_best_model_feature_importance.png", dpi = 300)
plt.show()

##  CONCLUSION & LIMITATIONS
Overall, the models were successful in understanding fraud factors leading to a high rate of fraud versus non-fraud detection. However, there are issues with the models and the dataset. As the dataset is small, certain patterns that were useful in prediction cannot be relied upon, as there is a high chance they do not generalise beyond the dataset. In particular, the `insured_hobbies` feature. While they provide valuable predictive power here, in a different dataset, a different set of hobbies could be correlated because of the limited sample size. These features may be correlated with fraud risk, but that would require a much larger dataset to prove. One positive is that the most important feature, `incident_severity_Major Damage`, is seen as directly linked to the fraud rate. This means that if features easily susceptible to sampling bias or unrelated to fraud are removed, it may be possible to still create a generalisable model.